# Ipotesi 1 - test dell'oracolo - LLaVA-Video 7B

Stesso protocollo di `01_videollama2_oracolo`: 49 domande a 6 opzioni, condizioni `O0_video` (video + oracolo)
e `O1_testo` (solo testo + oracolo), controllo `C0_controllo` sul baseline.


In [ ]:
# Runtime: Runtime > Cambia tipo di runtime > GPU L4, sessione NUOVA
import os, sys, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # meno frammentazione della VRAM
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"                    # evita il timeout del secret HF_TOKEN di Colab
os.environ.pop("HF_TOKEN", None)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Nessuna GPU: Runtime > Cambia tipo di runtime > L4"
print("GPU:", gpu, "| Python:", sys.version.split()[0])
if "L4" not in gpu:
    print("ATTENZIONE: il notebook e' tarato su L4. Un'altra GPU va bene se ha almeno 24 GB.")

# transformers 4.37.2: l'ultima compatibile con il codice di LLaVA-NeXT (come il baseline).
# Porta con se' tokenizers 0.15.2 e huggingface_hub < 1.0 (wheel anche per Python 3.13).
# accelerate serve per device_map="auto". peft non serve in inferenza e non si installa.
# torch resta quello di Colab, come nel baseline.
# Gli avvisi rossi di pip su gradio, sentence-transformers o peft sono attesi:
# sono pacchetti di Colab che questo notebook non importa.
%pip install -q "transformers==4.37.2" "accelerate>=0.29.1,<2" av timm einops sentencepiece

# Repo ufficiale LLaVA-NeXT, fissato a un commit: se il repo cambia, il codice resta quello del baseline
LLAVA_DIR = "/content/LLaVA-NeXT"
LLAVA_COMMIT = "bce12e479bc4dfee2b9c50c88137b01ff51bd483"   # ultimo commit del repo (giugno 2026)
%cd /content
if not os.path.exists(LLAVA_DIR):
    !git clone -q https://github.com/LLaVA-VL/LLaVA-NeXT.git {LLAVA_DIR}
!git -C {LLAVA_DIR} checkout -q {LLAVA_COMMIT}
!git -C {LLAVA_DIR} log -1 --format="LLaVA-NeXT: commit %h del %cd"
%cd /content/LLaVA-NeXT

# installo il pacchetto llava senza dipendenze (non deve toccare transformers)
%pip install -q -e ".[train]" --no-deps
if LLAVA_DIR not in sys.path:
    sys.path.insert(0, LLAVA_DIR)

from importlib.metadata import version
_vers = {k: version(k) for k in ["torch", "transformers", "tokenizers", "accelerate", "huggingface_hub"]}
print(_vers)
assert _vers["transformers"] == "4.37.2", f"transformers {_vers['transformers']}: controllare l'output di pip sopra"

GPU: NVIDIA L4, 23034 MiB | Python: 3.13.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.
/content
LLaVA-NeXT: commit bce12e4 del Mon Jun 15 14:32:49 2026 +0800
/content/LLaVA-NeXT
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting re

In [ ]:
import sys, torch, transformers
# Se nella sessione era gia' stato importato un altro transformers (es. il notebook VideoLLaMA 2),
# in memoria c'e' ancora quello: serve una sessione nuova.
assert transformers.__version__ == "4.37.2", (
    f"transformers {transformers.__version__} in memoria: Runtime > Disconnetti ed elimina runtime, poi da capo")
assert torch.cuda.is_available(), "CUDA non disponibile: il runtime non ha la GPU"

if "/content/LLaVA-NeXT" not in sys.path:
    sys.path.insert(0, "/content/LLaVA-NeXT")

# Loader ufficiale lmms-lab: carica pesi, tokenizer e image_processor
# con le chiavi corrette per LLaVA-Video-7B-Qwen2
from llava.model.builder import load_pretrained_model

model_path = "lmms-lab/LLaVA-Video-7B-Qwen2"
model_name = "llava_qwen"   # nome architettura richiesto dal builder

# carico il modello (circa 16 GB da scaricare, piu' la vision tower SigLIP)
tokenizer, model, image_processor, max_length = load_pretrained_model(
    model_path,
    None,           # model_base: None per modelli stand-alone
    model_name,
    torch_dtype="bfloat16",
    device_map="auto",
    attn_implementation="sdpa",
)

# mettiamo in modalita' evaluation
model.eval()

# Tutto il modello deve stare sulla GPU: con device_map="auto", se la memoria non basta,
# accelerate sposta dei layer sulla CPU senza errori ma rende l'inferenza lentissima.
fuori = {str(p.device) for p in model.parameters() if p.device.type != "cuda"}
assert not fuori, f"parte del modello e' su {fuori} (offload): serve piu' VRAM libera"
device = next(model.parameters()).device

# L4: logits solo per l'ultima posizione.
# transformers 4.37 nel prefill calcola i logits su TUTTA la sequenza (circa 6.900 token con
# 32 fotogrammi) e li converte in float32: 6.900 x 152.064 voci = circa 5,9 GiB in piu' oltre
# ai 15 GiB di pesi. Su A100 c'era spazio, su L4 (22 GiB utili) e' out of memory.
# Con decoding greedy si usa solo la riga dell'ultima posizione, quindi l'hook passa al lm_head
# solo quella. Gli hidden state sono gli stessi; puo' cambiare solo l'arrotondamento del
# prodotto finale, che conta solo se due lettere sono praticamente alla pari.
if globals().get("_hook_lm_head") is not None:
    _hook_lm_head.remove()
_hook_lm_head = model.lm_head.register_forward_pre_hook(lambda modulo, args: (args[0][:, -1:, :],))

print(f"LLaVA-Video-7B-Qwen2 caricato su {device}")
print(f"VRAM occupata: {torch.cuda.memory_allocated() / 1024**3:.1f} GiB "
      f"su {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")

/content/LLaVA-NeXT/llava/model/llava_arch.py:215: SyntaxWarning: "is not" with 'int' literal. Did you mean "!="?
  if slower_img_feat is not 0:


Please install pyav to use video processing functions.
OpenCLIP not installed
Loaded LLaVA model: lmms-lab/LLaVA-Video-7B-Qwen2


/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (14 outputs left out)

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:621: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:621: UserWarning: for vision_model.embeddings.patch_embedding.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:621: UserWarning: for vision_model.embeddings

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

Model Class: LlavaQwenForCausalLM
LLaVA-Video-7B-Qwen2 caricato su cuda:0
VRAM occupata: 15.4 GiB su 22.0 GiB


## 2. Percorsi e indice dei video

In [ ]:
import os, sys, glob, importlib
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Progetto Deep Learning 1"
TEST = os.path.join(BASE, "test_oracolo")        # hdepic_eval.py, ablazione.py, dati/, risultati/

# file che devono essere su Drive prima di iniziare
RICHIESTI = ["hdepic_eval.py", "ablazione.py",
             "dati/oracoli.json", "dati/nutrition_video_nutrition_estimation.json",
             "dati/annotazioni_oracolo/nutrition_video_nutrition_estimation.json"]
mancanti = [p for p in RICHIESTI if not os.path.exists(os.path.join(TEST, p))]
assert not mancanti, f"Mancano su Drive, in {TEST}: {mancanti}"
os.makedirs(os.path.join(TEST, "risultati"), exist_ok=True)

os.environ["TEST_ORACOLO_DIR"] = TEST            # ablazione.py legge da qui
if TEST not in sys.path:
    sys.path.insert(0, TEST)

import hdepic_eval as H
import ablazione as A
importlib.reload(H); importlib.reload(A)         # se la cella viene rieseguita

MODELLO = "llavavideo"
VIDEO_ROOTS = sorted(d for d in glob.glob(os.path.join(BASE, "*"))
                     if os.path.isdir(d) and "panel" not in os.path.basename(d).lower()
                     and os.path.basename(d).lower().startswith(("video", "videos")))
assert VIDEO_ROOTS, f"Nessuna cartella video* in {BASE}"
VIDEO_INDEX = H.build_video_index(VIDEO_ROOTS)
READABLE = {}     # cache: aprire i video da Drive e' lento, si controlla una volta sola

print("cartelle video:", [os.path.basename(d) for d in VIDEO_ROOTS])
print(f"video indicizzati: {len(VIDEO_INDEX['sampled'])} campionati, {len(VIDEO_INDEX['full'])} integrali")
print("annotazioni a 6 opzioni presenti:", os.path.exists(A.ANNOTAZIONI_ORACOLO))
print("baseline C0 presente:", os.path.exists(os.path.join(A.RISULTATI, A.nome_file(MODELLO, "C0_video"))))

Mounted at /content/drive
cartelle video: ['Videos Nutrition Change Sampled', 'Videos Nutrition Estimation Sampled', 'videos_image_nutrition_estimation_sampled']
video indicizzati: 50 campionati, 0 integrali
annotazioni a 6 opzioni presenti: True
baseline C0 presente: True


## 3. Controllo dei file

In [ ]:
cov, dettagli = H.preflight(A.ANN_ORACOLO_DIR, VIDEO_INDEX, tasks=[A.TASK], cache=READABLE)
print(cov.to_string(index=False))
rotti = sorted({v for ids in dettagli[A.TASK].values() for v in ids})
if rotti:
    print("file mancanti o non decodificabili:", rotti)
utili = int(cov.loc[cov["task"] == A.TASK, "utilizzabili"].iloc[0])
assert utili > 0, "Nessuna domanda con un video leggibile: controllare le cartelle video su Drive"

                      task input  domande  utilizzabili  scartate file_inutilizzabili
video_nutrition_estimation video       49            49         0                   0
                    TOTALE             49            49         0                    


## 4. Funzioni di inferenza

- **`infer_video`**: stessi 32 fotogrammi, stesso preprocessing e stesso prompt del baseline.
- **`infer_testo`**: nessun token immagine e nessun tensore visivo. Serve a O1.

In [ ]:
import av, torch
import numpy as np
from PIL import Image
from llava.conversation import conv_templates
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.mm_utils import tokenizer_image_token

N_FRAMES = 32
MAX_NEW_TOKENS = 32

def sample_video_frames(video_path, n=N_FRAMES):
    """n fotogrammi uniformi con PyAV (come nel codice ufficiale di LLaVA-Video)."""
    container = av.open(video_path)
    total = container.streams.video[0].frames
    if total == 0:
        total = sum(1 for _ in container.decode(video=0))
        container.close()
        container = av.open(video_path)
    voluti = set(np.linspace(0, total - 1, min(n, total), dtype=int).tolist())
    frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i in voluti:
            frames.append(frame.to_ndarray(format="rgb24"))
        if len(frames) == len(voluti):
            break
    container.close()
    return frames

def _immagini(visual, n=N_FRAMES):
    """path mp4 o lista di path -> lista di PIL.Image (liste lunghe sottocampionate a n)."""
    if isinstance(visual, list):
        paths = visual
        if len(paths) > n:
            sel = np.linspace(0, len(paths) - 1, n, dtype=int)
            paths = [paths[i] for i in sel]
        return [Image.open(p).convert("RGB") for p in paths]
    return [Image.fromarray(f) for f in sample_video_frames(visual, n)]

def _genera(messaggio, video_tensor=None):
    conv = conv_templates["qwen_1_5"].copy()
    conv.append_message(conv.roles[0], messaggio)
    conv.append_message(conv.roles[1], None)
    input_ids = tokenizer_image_token(conv.get_prompt(), tokenizer, IMAGE_TOKEN_INDEX,
                                      return_tensors="pt").unsqueeze(0).to(device)
    visivo = {} if video_tensor is None else \
             dict(images=[video_tensor], image_sizes=None, modalities=["video"])
    with torch.no_grad():
        out = model.generate(input_ids, **visivo, do_sample=False, temperature=None,
                             max_new_tokens=MAX_NEW_TOKENS, use_cache=True)
    testo = tokenizer.decode(out[0], skip_special_tokens=False)
    marker = "<|im_start|>assistant\n"
    if marker in testo:
        return testo.split(marker)[-1].replace("<|im_end|>", "").strip()
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def infer_video(visual, prompt):
    video_tensor = image_processor.preprocess(_immagini(visual), return_tensors="pt")["pixel_values"]
    video_tensor = video_tensor.to(device=device, dtype=torch.bfloat16)
    return _genera(f"{DEFAULT_IMAGE_TOKEN}\n{prompt}", video_tensor)

def infer_testo(prompt):
    """O1: solo testo."""
    return _genera(prompt)

INFER = {"O0_video": infer_video, "O1_testo": A.solo_testo(infer_testo)}
print("condizioni pronte:", list(INFER), "| fotogrammi per domanda:", N_FRAMES)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


condizioni pronte: ['O0_video', 'O1_testo'] | fotogrammi per domanda: 32


## 6. Controllo Baseline

Rifa' 5 domande del baseline e le confronta con quelle salvate. Devono coincidere tutte.


In [ ]:
BASELINE_C0 = os.path.join(A.RISULTATI, A.nome_file(MODELLO, "C0_video"))
if not os.path.exists(BASELINE_C0):
    print(f"Controllo C0 SALTATO: manca {BASELINE_C0}.\n"
          "Copiare li' il .jsonl del baseline di video_nutrition_estimation e rilanciare questa cella.")
else:
    ctrl = A.esegui(MODELLO, "C0_controllo", infer_video, A.DATI, VIDEO_INDEX,
                    limit=5, readable_cache=READABLE)
    A.controllo_c0(os.path.join(A.RISULTATI, A.nome_file(MODELLO, "C0_controllo")), BASELINE_C0)

  50 domande utilizzabili


/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `None` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:397: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:407: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  [1/50] OK    0 | pred=3 gt=3 | leading_letter     | 'D'
  [2/50] KO    1 | pred=2 gt=0 | leading_letter     | 'C'
  [3/50] OK    2 | pred=4 gt=4 | leading_letter     | 'E'
  [4/50] OK    3 | pred=4 gt=4 | leading_letter     | 'E'
  [5/50] OK    4 | pred=1 gt=1 | leading_letter     | 'B'
Controllo C0: 4/5 risposte identiche al baseline
ATTENZIONE: l'ambiente non coincide col baseline. Prima di continuare, rifare C0 in questa sessione.


## 7. Run completi

In [ ]:
for c in INFER:
    df = A.esegui(MODELLO, c, INFER[c], A.ANN_ORACOLO_DIR, VIDEO_INDEX, readable_cache=READABLE)
    print(f"{c}: {len(df)} domande")

  49 domande utilizzabili
  [1/49] KO    0 | pred=0 gt=3 | leading_letter     | 'A'
  [2/49] KO    1 | pred=3 gt=0 | leading_letter     | 'D'
  [3/49] KO   10 | pred=2 gt=3 | leading_letter     | 'C'
  [4/49] OK   11 | pred=5 gt=5 | leading_letter     | 'F'
  [5/49] OK   12 | pred=1 gt=1 | leading_letter     | 'B'
  [6/49] KO   13 | pred=1 gt=0 | leading_letter     | 'B'
  [7/49] OK   14 | pred=0 gt=0 | leading_letter     | 'A'
  [8/49] KO   15 | pred=4 gt=5 | leading_letter     | 'E'
  [9/49] OK   16 | pred=1 gt=1 | leading_letter     | 'B'
  [10/49] KO   17 | pred=2 gt=1 | leading_letter     | 'C'
  [11/49] KO   18 | pred=2 gt=4 | leading_letter     | 'C'
  [12/49] OK   19 | pred=0 gt=0 | leading_letter     | 'A'
  [13/49] KO    2 | pred=4 gt=5 | leading_letter     | 'E'
  [14/49] OK   20 | pred=2 gt=2 | leading_letter     | 'C'
  [15/49] KO   21 | pred=2 gt=0 | leading_letter     | 'C'
  [16/49] OK   22 | pred=5 gt=5 | leading_letter     | 'F'
  [17/49] KO   23 | pred=1 gt=3 | leadi

## 8. Prima lettura

In [ ]:
A.confronta([os.path.join(A.RISULTATI, A.nome_file(MODELLO, c)) for c in INFER])

run                                                       n  oracolo      %           IC 95%  accur.
----------------------------------------------------------------------------------------------------
llavavideo_O0_video_video_nutrition_estimation.jsonl     49        8  16.3%            9-29%   51.0%
llavavideo_O1_testo_video_nutrition_estimation.jsonl     49       23  46.9%           34-61%   34.7%

Caso: 16.7%   soglia per 'risponde dal testo': 50%
